# Market Basket Analysis with RELIM
Dataset: Retail (88,162 transactions)

In [ ]:
using Pkg
Pkg.activate("..")
include("../src/algorithm/relim.jl")

transactions = load_spmf("../data/real_world/retail.txt")
println("Loaded ", length(transactions), " transactions.")

In [ ]:
minsup = 0.01  # 1% 
frequent_itemsets_array = relim(transactions, minsup; relative=true)
frequent_itemsets = Dict(sort(iset) => sup for (iset, sup) in frequent_itemsets_array)
println("Found ", length(frequent_itemsets), " frequent itemsets.")

In [ ]:
using Combinatorics

function proper_subsets(itemset)
    subsets = []
    for k in 1:(length(itemset)-1)
        append!(subsets, collect(combinations(itemset, k)))
    end
    return [sort(s) for s in subsets]
end

function generate_rules(frequent_itemsets, all_transactions, minconf)
    rules = []
    n = length(all_transactions)
    
    for (itemset, sup_xy) in frequent_itemsets
        if length(itemset) < 2
            continue
        end
        for antecedent in proper_subsets(itemset)
            consequent = setdiff(itemset, antecedent)
            if isempty(consequent)
                continue
            end
            
            sup_x = get(frequent_itemsets, antecedent, 0)
            sup_y = get(frequent_itemsets, consequent, 0)
            
            if sup_x > 0 && sup_y > 0
                conf = sup_xy / sup_x
                if conf >= minconf
                    lift = (sup_xy / n) / ((sup_x / n) * (sup_y / n))
                    push!(rules, (antecedent, consequent, sup_xy/n, conf, lift))
                end
            end
        end
    end
    return rules
end

rules = generate_rules(frequent_itemsets, transactions, 0.5)
sort!(rules, by = r -> r[5], rev=true)
top10 = rules[1:min(10, end)]

using Printf
println("Top 10 Association Rules (sorted by Lift):")
println(rpad("Antecedent", 20), " => ", rpad("Consequent", 20), " | ", rpad("Support", 10), " | ", rpad("Confidence", 12), " | ", "Lift")
println("-"^85)
for (ant, con, sup, conf, lift) in top10
    @printf("%s => %s | %.4f     | %.4f       | %.4f\n", rpad(string(ant), 20), rpad(string(con), 20), sup, conf, lift)
end